In [1]:
import os
import sys
import yaml
from datetime import datetime
from scholarly import scholarly, ProxyGenerator

In [5]:
def load_scholar_user_id() -> str:
    """Load the Google Scholar user ID from the configuration file."""
    config_file = "_data/socials.yml"
    if not os.path.exists(config_file):
        print(
            f"Configuration file {config_file} not found. Please ensure the file exists and contains your Google Scholar user ID."
        )
        sys.exit(1)
    try:
        with open(config_file, "r") as f:
            config = yaml.safe_load(f)
        scholar_user_id = config.get("scholar_userid")
        if not scholar_user_id:
            print(
                "No 'scholar_userid' found in the configuration file. Please add 'scholar_userid' to _data/socials.yml."
            )
            sys.exit(1)
        return scholar_user_id
    except yaml.YAMLError as e:
        print(
            f"Error parsing YAML file {config_file}: {e}. Please check the file for correct YAML syntax."
        )
        sys.exit(1)

In [ ]:
SCHOLAR_USER_ID: str = load_scholar_user_id()
OUTPUT_FILE: str = "_data/citations.yml"


In [ ]:
def get_scholar_citations() -> None:
    """Fetch and update Google Scholar citation data."""
    print(f"Fetching citations for Google Scholar ID: {SCHOLAR_USER_ID}")
    today = datetime.now().strftime("%Y-%m-%d")

    # Check if the output file was already updated today
    if os.path.exists(OUTPUT_FILE):
        try:
            with open(OUTPUT_FILE, "r") as f:
                existing_data = yaml.safe_load(f)
            if (
                existing_data
                and "metadata" in existing_data
                and "last_updated" in existing_data["metadata"]
            ):
                print(f"Last updated on: {existing_data['metadata']['last_updated']}")
                if existing_data["metadata"]["last_updated"] == today:
                    print("Citations data is already up-to-date. Skipping fetch.")
                    return
        except Exception as e:
            print(
                f"Warning: Could not read existing citation data from {OUTPUT_FILE}: {e}. The file may be missing or corrupted."
            )

    citation_data = {"metadata": {"last_updated": today}, "papers": {}}

    scholarly.set_timeout(15)
    scholarly.set_retries(3)
    try:
        author = scholarly.search_author_id(SCHOLAR_USER_ID)
        author_data = scholarly.fill(author)
    except Exception as e:
        print(
            f"Error fetching author data from Google Scholar for user ID '{SCHOLAR_USER_ID}': {e}. Please check your internet connection and Scholar user ID."
        )
        sys.exit(1)

    if not author_data:
        print(
            f"Could not fetch author data for user ID '{SCHOLAR_USER_ID}'. Please verify the Scholar user ID and try again."
        )
        sys.exit(1)

    if "publications" not in author_data:
        print(f"No publications found in author data for user ID '{SCHOLAR_USER_ID}'.")
        sys.exit(1)

    for pub in author_data["publications"]:
        try:
            pub_id = pub.get("pub_id") or pub.get("author_pub_id")
            if not pub_id:
                print(
                    f"Warning: No ID found for publication: {pub.get('bib', {}).get('title', 'Unknown')}. This publication will be skipped."
                )
                continue

            title = pub.get("bib", {}).get("title", "Unknown Title")
            year = pub.get("bib", {}).get("pub_year", "Unknown Year")
            citations = pub.get("num_citations", 0)

            print(f"Found: {title} ({year}) - Citations: {citations}")

            citation_data["papers"][pub_id] = {
                "title": title,
                "year": year,
                "citations": citations,
            }
        except Exception as e:
            print(
                f"Error processing publication '{pub.get('bib', {}).get('title', 'Unknown')}': {e}. This publication will be skipped."
            )

    # Compare new data with existing data
    if existing_data and existing_data.get("papers") == citation_data["papers"]:
        print("No changes in citation data. Skipping file update.")
        return

    try:
        with open(OUTPUT_FILE, "w") as f:
            yaml.dump(citation_data, f, width=1000, sort_keys=True)
        print(f"Citation data saved to {OUTPUT_FILE}")
    except Exception as e:
        print(
            f"Error writing citation data to {OUTPUT_FILE}: {e}. Please check file permissions and disk space."
        )
        sys.exit(1)


In [23]:
today = datetime.now().strftime("%Y-%m-%d")

if os.path.exists(OUTPUT_FILE):
		try:
				with open(OUTPUT_FILE, "r") as f:
						existing_data = yaml.safe_load(f)
				if (
						existing_data
						and "metadata" in existing_data
						and "last_updated" in existing_data["metadata"]
				):
						print(f"Last updated on: {existing_data['metadata']['last_updated']}")
						if existing_data["metadata"]["last_updated"] == today:
								print("Citations data is already up-to-date. Skipping fetch.")
								# return
		except Exception as e:
				print(
						f"Warning: Could not read existing citation data from {OUTPUT_FILE}: {e}. The file may be missing or corrupted."
				)

Last updated on: 2026-01-26


In [12]:
scholarly.set_timeout(15)
scholarly.set_retries(3)

In [14]:
SCHOLAR_USER_ID

'GEJHkg0AAAAJ'

In [ ]:
# ---------- Proxy setting ----------
pg = ProxyGenerator()
use_proxy = False


In [24]:
citation_data = {"metadata": {"last_updated": today}, "papers": {}}

In [13]:
if pg.FreeProxies():
        scholarly.use_proxy(pg)
        print("Trying free proxy...")
        # Test the availability of proxy
        try:
            author = scholarly.search_author_id(SCHOLAR_USER_ID)
            use_proxy = True
		
            print("Free proxy works, using it.")
        except Exception as e:
            print(f"Free proxy test failed: {e}")

Trying free proxy...
Free proxy test failed: 'NoneType' object has no attribute 'get'


In [16]:
author=autor

In [17]:
author

{'container_type': 'Author',
 'filled': ['basics'],
 'scholar_id': 'GEJHkg0AAAAJ',
 'source': <AuthorSource.AUTHOR_PROFILE_PAGE: 'AUTHOR_PROFILE_PAGE'>,
 'name': 'Antonio Montero',
 'url_picture': 'https://scholar.googleusercontent.com/citations?view_op=view_photo&user=GEJHkg0AAAAJ&citpid=3',
 'affiliation': 'Research assistant FMF-UL Ljubljana, SI',
 'interests': ['Symmetries of discrete objects'],
 'email_domain': '@fmf.uni-lj.si',
 'homepage': 'http://anteromontonio.github.io/',
 'citedby': 24}

In [18]:
author_data = scholarly.fill(author)

In [25]:
for pub in author_data["publications"]:
        try:
            pub_id = pub.get("pub_id") or pub.get("author_pub_id")
            if not pub_id:
                print(
                    f"Warning: No ID found for publication: {pub.get('bib', {}).get('title', 'Unknown')}. This publication will be skipped."
                )
                continue

            title = pub.get("bib", {}).get("title", "Unknown Title")
            year = pub.get("bib", {}).get("pub_year", "Unknown Year")
            citations = pub.get("num_citations", 0)

            print(f"Found: {title} ({year}) - Citations: {citations}")

            citation_data["papers"][pub_id] = {
                "title": title,
                "year": year,
                "citations": citations,
            }
        except Exception as e:
            print(
                f"Error processing publication '{pub.get('bib', {}).get('title', 'Unknown')}': {e}. This publication will be skipped."
            )

Found: Equivelar toroids with few flag-orbits (2021) - Citations: 6
Found: Voltage operations on maniplexes, polytopes and maps (2023) - Citations: 4
Found: Locally spherical hypertopes from generlised cubes (2020) - Citations: 4
Found: Proper locally spherical hypertopes of hyperbolic type (2022) - Citations: 3
Found: Regular polyhedra in the 3-torus: Dedicated to Enrique Rodriguez Castillo, a beloved friend and colleague whom we all are going to miss (2018) - Citations: 3
Found: Cayley extensions of maniplexes and polytopes (2025) - Citations: 2
Found: Chiral extensions of regular toroids (2025) - Citations: 1
Found: On the Schläfli symbol of chiral extensions of polytopes (2021) - Citations: 1
Found: Vertex-transitive graphs with small motion and transitive permutation groups with small minimal degree (2025) - Citations: 0
Found: Symmetries of voltage operations on polytopes, maps and maniplexes (2025) - Citations: 0
Found: Chiral extensions of toroids (2019) - Citations: 0


In [28]:
# Compare new data with existing data
if existing_data and existing_data.get("papers") == citation_data["papers"]:
		print("No changes in citation data. Skipping file update.")
		# return

try:
		with open(OUTPUT_FILE, "w") as f:
				yaml.dump(citation_data, f, width=1000, sort_keys=True)
		print(f"Citation data saved to {OUTPUT_FILE}")
except Exception as e:
		print(
				f"Error writing citation data to {OUTPUT_FILE}: {e}. Please check file permissions and disk space."
		)
		sys.exit(1)

Citation data saved to _data/citations.yml
